<a href="https://colab.research.google.com/github/ndvphuc07/PALM/blob/main/palmistry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip -q "/content/drive/MyDrive/Colab Notebooks/palm/TEN_FILE_CUA_BAN.zip" -d "/content/Kho_Anh"

unzip:  cannot find or open /content/drive/MyDrive/Colab Notebooks/palm/TEN_FILE_CUA_BAN.zip, /content/drive/MyDrive/Colab Notebooks/palm/TEN_FILE_CUA_BAN.zip.zip or /content/drive/MyDrive/Colab Notebooks/palm/TEN_FILE_CUA_BAN.zip.ZIP.


In [ ]:
import pandas as pd
import os
import shutil

# 1. Đọc file CSV từ Drive
# This line is likely failing because Google Drive is not mounted.
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/palm/HandInfo.csv')

# 2. Lọc chỉ lấy những ảnh chụp lòng bàn tay (palmar)
palmar_hands = df[df['aspectOfHand'].str.contains('palmar')]
image_names = palmar_hands['imageName'].tolist()

# 3. Định nghĩa đường dẫn
# Tùy vào cấu trúc file zip của Kaggle, ảnh thường nằm trực tiếp trong thư mục giải nén
# hoặc nằm trong một thư mục con tên là 'Hands'. Bạn thử đường dẫn này trước:
source_folder = '/content/Kho_Anh/Hands'

# Nếu code báo lỗi tìm không thấy thư mục nguồn, hãy đổi dòng trên thành:
# source_folder = '/content/Kho_Anh'

destination_folder = '/content/drive/MyDrive/Long_Ban_Tay'

if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# 4. Copy ảnh sang thư mục mới
count = 0
for img_name in image_names:
    src_path = os.path.join(source_folder, img_name)
    dst_path = os.path.join(destination_folder, img_name)

    if os.path.exists(src_path):
        shutil.copy(src_path, dst_path)
        count += 1

    # Copy 500 ảnh cho nhẹ
    if count >= 500:
        break

print(f"Đã copy thành công {count} ảnh lòng bàn tay vào thư mục: {destination_folder}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/palm/HandInfo.csv'

In [ ]:
import os
import shutil
import glob

# 1. Tìm file ZIP trong thư mục palm của bạn
zip_folder = '/content/drive/MyDrive/Colab Notebooks/palm'
zip_files = glob.glob(os.path.join(zip_folder, '*.zip'))

if not zip_files:
    print("❌ Lỗi: Không tìm thấy file .zip nào trong thư mục palm! Bạn kiểm tra lại xem đã tải lên thành công chưa nhé.")
else:
    zip_path = zip_files[0]
    print(f"✅ Đã tìm thấy file nén: {zip_path}")
    print("⏳ Đang giải nén... (Chờ khoảng 10-20 giây nhé)")

    # Giải nén thẳng vào sảnh chính của Colab cho tốc độ bàn thờ
    extract_dir = '/content/Kho_Anh_Goc'
    os.system(f'unzip -q "{zip_path}" -d "{extract_dir}"')

    # 2. Tạo thư mục đích trên Drive để bạn ngồi phân loại
    destination_folder = '/content/drive/MyDrive/Long_Ban_Tay'
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    # 3. Lục lọi tất cả các file .jpg vừa được bung ra
    print("🔍 Đang gom ảnh...")
    all_jpgs = []
    for root, dirs, files in os.walk(extract_dir):
        for file in files:
            if file.lower().endswith('.jpg'):
                all_jpgs.append(os.path.join(root, file))

    # 4. Copy bừa 800 tấm sang Drive cho bạn
    count = 0
    for src_path in all_jpgs:
        dst_path = os.path.join(destination_folder, os.path.basename(src_path))
        if not os.path.exists(dst_path):
            shutil.copy(src_path, dst_path)
        count += 1
        if count >= 800:
            break

    print(f"🎉 Xong! Đã chép thành công {count} bức ảnh vào thư mục: {destination_folder}")
    print("Bây giờ bạn mở Drive lên, vào folder Long_Ban_Tay là thấy ảnh để chia vào 3 hệ Đất, Nước, Lửa rồi!")

✅ Đã tìm thấy file nén: /content/drive/MyDrive/Colab Notebooks/palm/Hands.zip
⏳ Đang giải nén... (Chờ khoảng 10-20 giây nhé)
🔍 Đang gom ảnh...
🎉 Xong! Đã chép thành công 800 bức ảnh vào thư mục: /content/drive/MyDrive/Long_Ban_Tay
Bây giờ bạn mở Drive lên, vào folder Long_Ban_Tay là thấy ảnh để chia vào 3 hệ Đất, Nước, Lửa rồi!


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

print("🚀 Đang khởi động AI...")

# 1. KHAI BÁO ĐƯỜNG DẪN DỮ LIỆU
# LƯU Ý: Sửa đường dẫn này đúng với thư mục chứa 3 folder He_Dat, He_Nuoc, He_Lua của bạn
thu_muc_data = '/content/drive/MyDrive/Colab Notebooks/Palmistry_Data'

# 2. XỬ LÝ ẢNH (Phóng to, thu nhỏ cho đồng đều và chia tập train/validation)
# Vì mình có ít ảnh (90 ảnh), việc tự động xoay lật ảnh sẽ giúp AI học tốt hơn
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2, # Trích 20% ảnh ra để làm bài kiểm tra (test)
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

print("Đang đọc dữ liệu học...")
train_data = datagen.flow_from_directory(
    thu_muc_data,
    target_size=(150, 150), # Ép tất cả ảnh về chung 1 kích thước vuông
    batch_size=8,           # Học mỗi lần 8 ảnh
    class_mode='categorical',
    subset='training'
)

print("Đang đọc dữ liệu kiểm tra...")
val_data = datagen.flow_from_directory(
    thu_muc_data,
    target_size=(150, 150),
    batch_size=8,
    class_mode='categorical',
    subset='validation'
)

# 3. XÂY DỰNG BỘ NÃO CNN
model = Sequential([
    # Lớp trích xuất đặc trưng (nhìn đường nét, hình dáng)
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    # Lớp nơ-ron phân loại
    Flatten(),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax') # Output có 3 hệ: Đất, Nước, Lửa
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# 4. BẮT ĐẦU HUẤN LUYỆN (TRAIN)
print("🧠 Bắt đầu cho AI đi học (Sẽ mất khoảng 1-2 phút)...")
history = model.fit(
    train_data,
    epochs=15, # Học đi học lại 15 lần (vòng lặp)
    validation_data=val_data
)

print("🎉 XONG! Mô hình AI đã học xong cách xem bói tay!")

🚀 Đang khởi động AI...
Đang đọc dữ liệu học...
Found 72 images belonging to 3 classes.
Đang đọc dữ liệu kiểm tra...
Found 18 images belonging to 3 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


🧠 Bắt đầu cho AI đi học (Sẽ mất khoảng 1-2 phút)...
Epoch 1/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 474ms/step - accuracy: 0.2500 - loss: 9.0165 - val_accuracy: 0.3333 - val_loss: 2.2151
Epoch 2/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 424ms/step - accuracy: 0.3889 - loss: 1.2067 - val_accuracy: 0.3333 - val_loss: 1.0991
Epoch 3/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 656ms/step - accuracy: 0.3194 - loss: 1.0988 - val_accuracy: 0.3333 - val_loss: 1.0988
Epoch 4/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 415ms/step - accuracy: 0.3333 - loss: 1.0988 - val_accuracy: 0.3333 - val_loss: 1.0986
Epoch 5/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 455ms/step - accuracy: 0.2917 - loss: 1.1012 - val_accuracy: 0.3333 - val_loss: 1.0985
Epoch 6/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 647ms/step - accuracy: 0.3333 - loss: 1.0988 - val_accuracy: 0.3333 - val_loss: 1.0985
Epoch 7/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 403ms/step - accuracy: 0.3472 - loss: 1.0986 - val_accuracy: 0.3889 - val_loss: 1.0991
Epoch 8/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 4s 403ms/step - accuracy: 0.27

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image

# Keras sắp xếp theo A-Z: He_Dat (0), He_Lua (1), He_Nuoc (2)
class_names = ['Bàn tay hệ Đất 🌍', 'Bàn tay hệ Lửa 🔥', 'Bàn tay hệ Nước 💧']

# BÍ KÍP TRỊ DARK MODE: Khóa cứng màu chữ bằng mã HEX và webkit
bold_black = "font-weight: 800; color: #000000 !important; -webkit-text-fill-color: #000000 !important;"
normal_black = "color: #000000 !important; -webkit-text-fill-color: #000000 !important;"

phan_tich_he = {
    0: {
        "tinh_duyen": f"❤️ <span style='{bold_black}'>Tình duyên:</span> <span style='{normal_black}'>Chung thủy, thực tế và chân thành. Các hạ không thích sự màu mè hay vội vã. Khi đã yêu là muốn gắn bó lâu dài.</span>",
        "tri_tue": f"🧠 <span style='{bold_black}'>Trí tuệ:</span> <span style='{normal_black}'>Tư duy logic, rõ ràng và cẩn trọng. Các hạ làm việc có kế hoạch, thích sự ổn định.</span>",
        "sinh_menh": f"🧬 <span style='{bold_black}'>Sinh mệnh:</span> <span style='{normal_black}'>Căn cơ vững vàng, sức khỏe dẻo dai. Các hạ có sức chịu đựng rất tốt trước sóng gió.</span>"
    },
    1: {
        "tinh_duyen": f"❤️ <span style='{bold_black}'>Tình duyên:</span> <span style='{normal_black}'>Nồng nhiệt, mãnh liệt và yêu ghét cực kỳ rõ ràng. Các hạ thích sự chủ động, cuốn hút.</span>",
        "tri_tue": f"🧠 <span style='{bold_black}'>Trí tuệ:</span> <span style='{normal_black}'>Quyết đoán, sáng tạo và đầy tham vọng. Các hạ xử lý vấn đề nhanh nhạy, thích dẫn dắt.</span>",
        "sinh_menh": f"🧬 <span style='{bold_black}'>Sinh mệnh:</span> <span style='{normal_black}'>Tràn đầy năng lượng và nhiệt huyết. Cuộc sống bận rộn, năng động.</span>"
    },
    2: {
        "tinh_duyen": f"❤️ <span style='{bold_black}'>Tình duyên:</span> <span style='{normal_black}'>Lãng mạn, nhạy cảm và giàu lòng trắc ẩn. Các hạ biết lắng nghe, dễ đồng cảm.</span>",
        "tri_tue": f"🧠 <span style='{bold_black}'>Trí tuệ:</span> <span style='{normal_black}'>Trực giác cực kỳ nhạy bén, thiên về cảm xúc và nghệ thuật. Các hạ giải quyết vấn đề uyển chuyển.</span>",
        "sinh_menh": f"🧬 <span style='{bold_black}'>Sinh mệnh:</span> <span style='{normal_black}'>Dòng chảy sinh mệnh êm đềm nhưng sâu sắc. Sức khỏe phụ thuộc nhiều vào trạng thái tinh thần.</span>"
    }
}

def du_doan_bo_tay(img):
    # Tiền xử lý ảnh
    img = img.resize((150, 150))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0

    # AI dự đoán
    predictions = model.predict(img_array)
    predicted_class_index = np.argmax(predictions[0])
    do_tu_tin = predictions[0][predicted_class_index] * 100

    ket_qua = class_names[predicted_class_index]
    loi_phan = phan_tich_he[predicted_class_index]

    # Định dạng kết quả đầu ra HTML/CSS
    html_output = f"""
    <div style="background: linear-gradient(135deg, #fdfbfb 0%, #ebedee 100%);
                padding: 25px;
                border-radius: 15px;
                box-shadow: 0 4px 6px rgba(0,0,0,0.1);
                text-align: center;
                border: 2px solid #d4af37;
                max-width: 450px;
                margin: 0 auto;">
        <h3 style="color: #6b7280; font-size: 16px; margin-bottom: 5px;">✨ Quẻ bói của các hạ là ✨</h3>
        <h1 style="color: #b91c1c; font-size: 30px; font-weight: bold; margin: 0;">{ket_qua}</h1>
        <p style="font-size: 15px; margin-top: 10px;"><span style='{normal_black}'>Đạo hạnh (Độ tự tin):</span> <span style='{bold_black}'>{do_tu_tin:.2f}%</span></p>

        <hr style="border-top: 1px dashed #cbd5e1; margin: 15px 0;">

        <div style="text-align: left; font-size: 15px; line-height: 1.6; padding: 0 10px;">
            <p style="margin-bottom: 12px; margin-top: 0;">{loi_phan['tinh_duyen']}</p>
            <p style="margin-bottom: 12px; margin-top: 0;">{loi_phan['tri_tue']}</p>
            <p style="margin-bottom: 0; margin-top: 0;">{loi_phan['sinh_menh']}</p>
        </div>
    </div>
    """
    return html_output

# Sử dụng Theme Soft
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="purple",
    font=[gr.themes.GoogleFont("Be Vietnam Pro"), "sans-serif"]
)

# Xây dựng bố cục với gr.Blocks
with gr.Blocks(theme=custom_theme, title="Palmora - AI Thần Toán") as app:
    # Phần Tiêu đề
    gr.HTML("""
        <div style="text-align: center; max-width: 800px; margin: 0 auto; padding-bottom: 20px;">
            <h1 style="color: #4338ca; font-size: 40px; font-weight: bold;">📜 Palmora 📜</h1>
            <p style="font-size: 18px; color: #64748b;">
                Tại hạ đã luyện thành thuật toán Mạng Nơ-ron (CNN). Hãy đưa tay đây, để tại hạ xem căn cơ và tiền trình của các hạ!
            </p>
        </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="Bức họa lòng bàn tay")
            btn_predict = gr.Button("🔮 Gieo Quẻ Ngay 🔮", variant="primary", size="lg")

        with gr.Column(scale=1):
            output_html = gr.HTML(label="Lời phán")

    btn_predict.click(fn=du_doan_bo_tay, inputs=input_image, outputs=output_html)

app.launch(share=True)

/tmp/ipykernel_2903/4165230175.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, title="Palmora - AI Thần Toán") as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2da96cd3f31c971445.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
